# Rooftop Solar - Inria training on Kaggle GPU

Trains a **general** rooftop-detection model on the full Inria Aerial Image Labeling dataset (180 tiles, 5000x5000 @ 0.3 m/px, five cities), mounted from the public Kaggle copy - no upload. Labels are building footprints; the usable-for-PV fraction is a downstream packing factor.

Split: Inria's official protocol (tiles 1-5 -> val, 6-36 -> train), so val IoU is comparable to published work (~0.78-0.82). Target: >= 0.72.

## 0. GPU check, before importing torch

In [ ]:
import subprocess, sys, os, json, time, shutil
from pathlib import Path

def sh(cmd):
    try:
        r = subprocess.run(cmd, capture_output=True, text=True)
        return (r.stdout or r.stderr).strip()
    except FileNotFoundError:
        return "<" + cmd[0] + " not found>"

gpu_name = sh(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"])
print("GPU:", gpu_name)
if "not found" in gpu_name:
    raise SystemExit("No GPU. Push with: kaggle kernels push -p kaggle_joint --accelerator gpuT4x2")

PROBE = (
    "import torch;"
    "p=torch.cuda.get_device_properties(0);"
    "cap='sm_'+str(p.major)+str(p.minor);"
    "t=torch.randn(256,256,device='cuda');"
    "ok=bool(torch.isfinite(t@t).all());"
    "print(torch.__version__, cap, cap in torch.cuda.get_arch_list(), ok)"
)

def probe():
    r = subprocess.run([sys.executable, "-c", PROBE], capture_output=True, text=True)
    return r.stdout.strip(), r.returncode, r.stderr.strip()[-400:]

out, rc, err = probe()
print("probe:", out or err)
if rc != 0 or "True True" not in out:
    print("this torch cannot drive this GPU - installing a cu118 build")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "torch==2.4.1+cu118", "torchvision==0.19.1+cu118",
         "--index-url", "https://download.pytorch.org/whl/cu118"],
        capture_output=True, text=True)
    print(r.stdout[-1200:] if r.returncode == 0 else r.stderr[-2000:])
    if r.returncode != 0:
        raise SystemExit("cu118 torch install failed")
    out, rc, err = probe()
    print("probe after reinstall:", out or err)
if rc != 0 or "True True" not in out:
    raise SystemExit("GPU still unusable (" + (out or err) + "). Try --accelerator gpuT4x2")
print("GPU verified by real matmul in a fresh interpreter")

## 1. Code + data

The repo is cloned for the code; Inria is mounted via `dataset_sources` and located under `/kaggle/input`.

In [ ]:
REPO = "https://github.com/Parthesh10/rooftop-solar-potential-detection.git"
WORK = Path("/kaggle/working")
SRC = WORK / "repo"

if SRC.exists():
    shutil.rmtree(SRC)
subprocess.run(["git", "clone", "--depth", "1", REPO, str(SRC)], check=True)
os.chdir(SRC)
sys.path.insert(0, str(SRC))
print("cloned at", sh(["git", "rev-parse", "--short", "HEAD"]))

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "tifffile", "nvidia-ml-py",
                "segmentation-models-pytorch>=0.5.0"], check=False)
import segmentation_models_pytorch as smp
print("segmentation-models-pytorch", smp.__version__)

# Find the mounted Inria dataset. dataset_sources mounts it under /kaggle/input;
# don't hardcode the exact nesting - just locate AerialImageDataset.
INRIA = None
for base in Path("/kaggle/input").glob("**/AerialImageDataset"):
    if (base / "train" / "images").is_dir():
        INRIA = base
        break
if INRIA is None:
    raise SystemExit(
        "Inria not mounted. Add it in the notebook's Data panel or set "
        "dataset_sources=['sagar100rathod/inria-aerial-image-labeling-dataset'] in kernel-metadata.json")
n_train = len(list((INRIA / "train" / "images").glob("*.tif")))
print("Inria at", INRIA, "-", n_train, "train tiles")

# The Indian tiles: images/ + labels/ pairs, uploaded as a Kaggle dataset
# because data/ is gitignored in the repo.
# How many tiles the dataset is supposed to contain. The first joint run
# trained on 16 of them instead of all of them and said nothing: the kernel was
# pushed ~2 minutes after the dataset was created, so Kaggle was still
# processing the upload and mounted a partial copy. The India half of the run
# was therefore ~5% of an epoch instead of ~22%, and the only visible symptom
# was a muted result. Assert the count rather than trust the mount.
EXPECTED_EXTRA_TILES = 1325

EXTRA = None
best_n = 0
for base in sorted(Path("/kaggle/input").glob("**/images")):
    labels = base.parent / "labels"
    if not labels.is_dir() or "AerialImage" in str(base):
        continue
    n = len(list(base.glob("*.png")))
    # Take the richest match, not the first: glob order is not meaningful and a
    # partial or stray directory must not win by being earlier.
    if n > best_n:
        EXTRA, best_n = base.parent, n
if EXTRA is None:
    raise SystemExit(
        "Indian tiles not mounted. Add partheshgupta/rooftop-solar-indian-osm-tiles in the Data panel or "
        "in dataset_sources.")

n_lbl = len(list((EXTRA / "labels").glob("*_label.png")))
print("Indian tiles at", EXTRA, "-", best_n, "images,", n_lbl, "labels")
if best_n < EXPECTED_EXTRA_TILES or n_lbl < EXPECTED_EXTRA_TILES:
    raise SystemExit(
        "Only " + str(best_n) + " images / " + str(n_lbl) + " labels mounted, "
        "expected " + str(EXPECTED_EXTRA_TILES) + ". The dataset is probably "
        "still processing - wait for it to finish on the dataset page, then "
        "re-push. Training on a partial mount silently under-weights the new "
        "region, which is exactly the failure this check exists to catch.")

## 2. Train

Windowed 512x512 reads straight from the GeoTIFFs (`process_data.inria.InriaWindowDataset`) - no pre-cutting to disk. AMP `auto` -> fp16 on a T4. Checkpoints every 5 min and every epoch, so `scripts/train_inria.py --resume` continues in a second session if the 12 h cap is hit.

In [ ]:
RUNS = WORK / "runs"
RUNS.mkdir(exist_ok=True)
os.environ["RUNS_ROOT"] = str(RUNS)

# ROUND 3 — a deliberate single-variable experiment. Only BREADTH changes.
#
# v2 moved two things at once: tiles 103 -> 505 and share 22% -> 35%. It worked
# (six of six held-out cities improved, Inria cost 0.0022), but with two
# variables moving, "which one mattered" stayed open. Fact 36 says it was the
# places. This run tests exactly that claim and nothing else.
#
# Inria contributes 155 x 48 = ~7.4k windows an epoch. 1325 tiles at
# --extra-repeat 3 is ~3.98k, i.e. **34.8% of each epoch against v2's 35.2%** —
# the same share, reached with 2.6x more distinct tiles and less repetition.
# Anything that moves is attributable to the data being broader, not heavier.
#
# The breadth: 25 AOIs on four continents, where v2 had ten cities in one
# country. Africa (Lagos, Nairobi, Accra, Addis), Southeast Asia (Dhaka,
# Jakarta, Manila), Latin America (Sao Paulo, Lima), plus eight Indian tier-2
# cities chosen because the weakest held-out scores (Ahmedabad 0.414, Jaipur
# 0.353) were the least metro-like of the set.
SWEEP = [
    ("J3_joint_world25", ["--arch", "unet++", "--encoder", "efficientnet-b0",
                          "--samples-per-tile", "48", "--epochs", "50",
                          "--extra-data-dir", str(EXTRA),
                          "--extra-ignore-value", "128",
                          "--extra-repeat", "3"],
     "U-Net++ / EfficientNet-B0 on Inria + 1325 tiles from 25 AOIs on four continents"),
]

BASE = [sys.executable, "-u", "scripts/train_inria.py",
        "--inria-root", str(INRIA),
        "--encoder-weights", "imagenet",
        "--window", "512", "--val-stride", "512",
        "--pos-weight", "2.4", "--dice-weight", "0.6",
        "--batch-size", "16", "--lr", "3e-4",
        # fp16 explicitly, not "auto": select_amp skips AMP on any card without
        # tensor cores, a rule written for a GTX 1650, and Kaggle hands out P100s
        # that do have fast packed fp16. The NaN probe still guards it.
        "--workers", "2", "--patience", "12", "--amp", "fp16",
        "--data-parallel",
        "--gpu-util-target", "100", "--gpu-temp-limit", "0",
        "--gpu-mem-fraction", "0.95", "--checkpoint-every", "300",
        "--no-progress"]

t_all = time.time()
for name, extra, note in SWEEP:
    print("")
    print("=" * 72)
    print("RUN " + name + "  -  " + note)
    print("=" * 72, flush=True)
    cmd = BASE + ["--run-name", name] + extra
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    print(name + " exit=" + str(proc.returncode) +
          "  " + str(round((time.time() - t0) / 60, 1)) + " min", flush=True)

print("")
print("total " + str(round((time.time() - t_all) / 60, 1)) + " min")

## 3. Collect artifacts

In [ ]:
summaries = []
for d in sorted(RUNS.iterdir()):
    f = d / "summary.json"
    if d.is_dir() and f.exists():
        summaries.append(json.loads(f.read_text()))

print("run              arch / encoder             ep  best_ep   VAL IoU     P      R")
print("-" * 78)
best = None
for s in sorted(summaries, key=lambda x: -x["best_val_iou"]):
    c = s["config"]
    m = s["metrics"].get("val", {})
    desc = c.get("arch", "unet") + " / " + str(c.get("encoder"))
    print(s["run"].ljust(16) + desc.ljust(27) +
          str(s["epochs_run"]).rjust(3) + str(s["best_epoch"]).rjust(8) +
          ("%.4f" % s["best_val_iou"]).rjust(10) +
          ("%.3f" % m.get("precision", float("nan"))).rjust(7) +
          ("%.3f" % m.get("recall", float("nan"))).rjust(7))
    if best is None:
        best = s

print("")
print("published Inria building-seg work: ~0.78-0.82 IoU;  plan.md target: >= 0.72")
if best:
    print("best here: VAL IoU " + ("%.4f" % best["best_val_iou"]) + "  (" + best["run"] + ")")

# Ship the winning weights + every summary; drop the repo clone and resume states.
if best:
    shutil.copy2(RUNS / best["run"] / "best.pt", WORK / "best_inria.pt")
    (WORK / "sweep_inria.json").write_text(json.dumps(summaries, indent=2))
    for f in ("history.json", "train.log", "metadata.json"):
        if (RUNS / best["run"] / f).exists():
            shutil.copy2(RUNS / best["run"] / f, WORK / (best["run"] + "_" + f))
h = json.loads((RUNS / best["run"] / "history.json").read_text()) if best else None

for d in RUNS.iterdir():
    if d.is_dir():
        for junk in ("state.pt", "state.pt.bak", "state.pt.tmp", "last.pt", "best.pt"):
            (d / junk).unlink(missing_ok=True)
os.chdir(WORK)
shutil.rmtree(SRC, ignore_errors=True)
shutil.rmtree(RUNS, ignore_errors=True)
print("")
print("artifacts:", sorted(q.name for q in WORK.glob("*") if q.is_file()))

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if h:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(h["train_loss"], label="train")
    ax[0].plot(h["val_loss"], label="val")
    ax[0].set_title("loss")
    ax[1].plot(h["train_iou"], label="train")
    ax[1].plot(h["val_iou"], label="val")
    if h.get("best_epoch") is not None:
        ax[1].axvline(h["best_epoch"], ls="--", c="k", lw=0.8)
    ax[1].set_title("IoU  (Inria official val)")
    for a in ax:
        a.set_xlabel("epoch")
        a.legend()
        a.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(WORK / "curves_inria.png", dpi=130)
    plt.show()